In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from scipy.stats import t
from statsmodels.regression.linear_model import OLS
from scipy.stats import spearmanr 
import statsmodels.api as sm
from sklearn.linear_model import Ridge

In [3]:
col_factor_qualified = ["z_vol_14d","z_vol_of_vol_14d"]
col_forward_return = ["fwd_1d", "fwd_5d", "fwd_14d"]

df = pd.read_parquet("../data/processed/2022-2025_factor_construction.parquet")
df

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d,fwd_1d,fwd_5d,fwd_14d
0,2022-01-01 00:00:00+00:00,BTC/USDT,1,-0.218301,-0.518497,0.023830,-0.266958,-0.394565,-0.471287,-1.277533,-1.382781,-1.798911,-1.080754,0.032060,-1.290804,-1.346109,-0.009188,-0.102294,-0.102248
1,2022-01-01 00:00:00+00:00,ETH/USDT,1,-0.532647,-0.912041,-0.024534,-0.068872,-0.629480,-0.321926,-1.254325,-1.499076,-1.678966,0.177704,0.024004,-1.282603,-1.339762,0.016522,-0.100115,-0.124109
2,2022-01-01 00:00:00+00:00,BNB/USDT,1,-0.033832,-0.696279,0.050409,0.127435,-0.436623,-0.503599,-1.218935,-1.584325,-1.870931,-0.175895,0.030618,-1.214010,-1.270937,0.006992,-0.109520,-0.064222
3,2022-01-01 00:00:00+00:00,LUNA/USDT,1,-0.380914,0.577266,1.980223,1.164109,0.722353,0.426418,0.071524,0.036502,0.934555,1.591304,0.070365,-1.118527,-1.121848,-0.024734,-0.156462,-0.050324
4,2022-01-01 00:00:00+00:00,SAND/USDT,1,-0.871395,0.208798,0.239387,3.030310,-0.691922,-0.447297,-0.743568,1.646163,0.647980,1.415158,0.021862,-1.042241,-1.102369,-0.009780,-0.131208,-0.209213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317671,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003923,NaN,NaN,0.024672,0.105572,0.146774
317672,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000985,NaN,NaN,0.012474,0.084218,0.092330
317673,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007752,NaN,NaN,0.017225,0.055387,0.042519
317674,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.015152,NaN,NaN,0.073563,0.161755,0.094616


# Phần D : Composite Alpha — Linear Baselines vs ML

In [4]:
# Chuyển đối dấu của từng factor mà ta đã thực hiện ở cuối phần C, để có xu hướng z_score tăng thì forward_return cũng tăng
d1_df = df[["timestamp","symbol","in_universe"] + col_factor_qualified + col_forward_return].copy()

d1_df["z_vol_14d"] = -1 * d1_df["z_vol_14d"]
d1_df["z_vol_of_vol_14d"] = -1 * d1_df["z_vol_of_vol_14d"]

d1_df


,timestamp,symbol,in_universe,z_vol_14d,z_vol_of_vol_14d,fwd_1d,fwd_5d,fwd_14d
0,2022-01-01 00:00:00+00:00,BTC/USDT,1,1.382781,1.080754,-0.009188,-0.102294,-0.102248
1,2022-01-01 00:00:00+00:00,ETH/USDT,1,1.499076,-0.177704,0.016522,-0.100115,-0.124109
2,2022-01-01 00:00:00+00:00,BNB/USDT,1,1.584325,0.175895,0.006992,-0.109520,-0.064222
3,2022-01-01 00:00:00+00:00,LUNA/USDT,1,-0.036502,-1.591304,-0.024734,-0.156462,-0.050324
4,2022-01-01 00:00:00+00:00,SAND/USDT,1,-1.646163,-1.415158,-0.009780,-0.131208,-0.209213
...,...,...,...,...,...,...,...,...
317671,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,0.024672,0.105572,0.146774
317672,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,0.012474,0.084218,0.092330
317673,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,0.017225,0.055387,0.042519
317674,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,0.073563,0.161755,0.094616


In [ ]:
"""
Tạo ra từng khoảng thời gian để TRAIN và khoảng để TEST
"""
def create_walk_forward_folds(train_window_months=12,test_window_months=1,research_start='2023-01-01',
                              research_end='2025-12-31',data_start='2022-01-01'):
    """
    Returns:
    --------
    folds : list of dict
        Mỗi dict chứa: fold_id, train_start, train_end, test_start, test_end
    """
    
    # Chuyển sang datetime
    research_start = pd.to_datetime(research_start)
    research_end = pd.to_datetime(research_end)
    data_start = pd.to_datetime(data_start)
    
    
    test_months = pd.date_range(
        start=research_start,
        end=research_end,
        freq='MS'  # Monthly Start (ngày đầu mỗi tháng)
    )
    
    folds = []
    
    for i, test_start in enumerate(test_months):
        test_end = test_start + pd.offsets.MonthEnd(0)  # Ngày cuối tháng
        
        # Training period: 12 tháng trước test_start
        train_end = test_start - pd.Timedelta(days=1)  # Ngày cuối cùng trước test
        train_start = train_end - pd.DateOffset(months=train_window_months - 1)
        train_start = train_start.replace(day=1)  # Ngày đầu tháng
        
        # Kiểm tra train_start >= data_start
        if train_start < data_start:
            # Nếu không đủ dữ liệu, bỏ qua fold này (hoặc điều chỉnh)
            continue
        
        folds.append({
            'fold_id': i + 1,
            'train_start': train_start,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end
        })
    
    return pd.DataFrame(folds)

fold = create_walk_forward_folds()
fold

,fold_id,train_start,train_end,test_start,test_end
0,1,2022-01-01,2022-12-31,2023-01-01,2023-01-31
1,2,2022-02-01,2023-01-31,2023-02-01,2023-02-28
2,3,2022-03-01,2023-02-28,2023-03-01,2023-03-31
3,4,2022-04-01,2023-03-31,2023-04-01,2023-04-30
4,5,2022-05-01,2023-04-30,2023-05-01,2023-05-31
5,6,2022-06-01,2023-05-31,2023-06-01,2023-06-30
6,7,2022-07-01,2023-06-30,2023-07-01,2023-07-31
7,8,2022-08-01,2023-07-31,2023-08-01,2023-08-31
8,9,2022-09-01,2023-08-31,2023-09-01,2023-09-30
9,10,2022-10-01,2023-09-30,2023-10-01,2023-10-31


## Phần D1 : Composite Alpha — Linear Baselines 

Ta sẽ xây dựng alpha theo 2 hướng trước, đây là cách xây dựng baseline để so sánh với các cách xây dựng alpha phía sau
- Equal_weight: ứng với ngày t, của con x, ta thực hiện tính trung bình cộng của các factor z_
- Evidence_weight:
    - Gỉả sử ta đang ở fold 1, ta sẽ sử dụng dữ liệu từ train_start- train_end để tính ra ic_mean của từng factor, và weight cố định của từng factor là ic_mean của factor đó / tổng các ic_mean của tất cả factor
    - Với ngày t, coint x, ta sẽ lấy weight đã có định nhân với z_score của factor tương ứng và tính tổng các factor đó lại
    - ví dụ weight của factor A,B lần lượt là x,y được tính từ dữ liệu đã định. Ngày t, coin bất kỳ có z_score của 2 factor lần luot là a,b thì evidence_weight = a*x + b*y


In [ ]:


# copy lại hàm từ 2_statistical_significant_factor
def build_daily_ic(df, col_factor_qualified, col_forward_return):
    in_universe = df[df["in_universe"] == 1].copy()
    
    groups = in_universe.groupby("timestamp")
    result = []
    
    for date, group in groups:
        factor_data = group[col_factor_qualified]
        foward_data = group[col_forward_return]
        
        for i, factor in enumerate(col_factor_qualified):
            factor_value = factor_data.loc[:,factor]
            
            for j, foward in enumerate(col_forward_return):
                foward_value = foward_data.loc[:,foward]
                
                corr, p_value = spearmanr(factor_value, foward_value)
                result.append({
                    "timestamp": date,
                    "factor_id": factor,
                    "horizon":foward,
                    "ic_spearmanr": corr  
                })
                
                
    ic_daily = pd.DataFrame(result)
    return ic_daily


In [7]:
ic_daily = build_daily_ic(d1_df, col_factor_qualified, col_forward_return)
ic_daily


In [ ]:
def calculate_ic_period(df,ic_daily:pd.DataFrame ,fold: pd.DataFrame, horizon: str, col_factor_qualified):
    
    ic_daily["timestamp"] = pd.to_datetime(ic_daily["timestamp"],utc=True)

    fold["train_start"] = pd.to_datetime(fold["train_start"],utc=True)

    fold["train_end"] = pd.to_datetime(fold["train_end"],utc=True)
    
    col_factor_ic = []
    for factor in col_factor_qualified:
        fold[f"ic_mean_{factor}"] = None
        col_factor_ic.append(f"ic_mean_{factor}")
        
    for f in fold.index:
        train_start = fold.loc[f, "train_start"]
        train_end = fold.loc[f, "train_end"]
       
        ic_period = ic_daily[(ic_daily["timestamp"] >= train_start ) & (ic_daily["timestamp"] <= train_end)& 
                             (ic_daily["horizon"] == horizon)].copy()
        group = ic_period.groupby("factor_id").agg(ic_mean = ("ic_spearmanr", "mean"))
        for factor in col_factor_qualified:
            value = group.loc[factor, "ic_mean"]

            fold.loc[f, f"ic_mean_{factor}"] = value
    
    total = fold[col_factor_ic].sum(axis= 1)
    for factor in col_factor_qualified:
        fold[f"weight_{factor}"] = fold[f"ic_mean_{factor}"] / total

    return fold

fold_ic = calculate_ic_period(d1_df, ic_daily, fold, "fwd_1d", col_factor_qualified) 
fold_ic.head()

,fold_id,train_start,train_end,test_start,test_end,ic_mean_z_vol_14d,ic_mean_z_vol_of_vol_14d,weight_z_vol_14d,weight_z_vol_of_vol_14d
0,1,2022-01-01 00:00:00+00:00,2022-12-31 00:00:00+00:00,2023-01-01,2023-01-31,0.123696,0.079029,0.610166,0.389834
1,2,2022-02-01 00:00:00+00:00,2023-01-31 00:00:00+00:00,2023-02-01,2023-02-28,0.104849,0.061353,0.630855,0.369145
2,3,2022-03-01 00:00:00+00:00,2023-02-28 00:00:00+00:00,2023-03-01,2023-03-31,0.103684,0.064275,0.617316,0.382684
3,4,2022-04-01 00:00:00+00:00,2023-03-31 00:00:00+00:00,2023-04-01,2023-04-30,0.11055,0.063878,0.633786,0.366214
4,5,2022-05-01 00:00:00+00:00,2023-04-30 00:00:00+00:00,2023-05-01,2023-05-31,0.10291,0.053385,0.658435,0.341565


In [9]:
def build_composite_alpha(df, test_start, test_end,fold_ic ,col_factor_qualified, horizon:str):
    test_start = pd.to_datetime(test_start, utc=True)
    test_end = pd.to_datetime(test_end, utc=True)
    fold_ic = fold_ic.copy()
    fold_ic["test_start"] = pd.to_datetime(fold_ic["test_start"], utc=True)
    fold_ic["test_end"] = pd.to_datetime(fold_ic["test_end"], utc=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    

    temp = df[(df["timestamp"] >= test_start) & (df["timestamp"] <= test_end)].copy() 
    
    df_period = temp[["timestamp","symbol","in_universe"] + col_factor_qualified + [horizon]]
    
    which_fold = fold_ic[(fold_ic["test_start"] == test_start) & (fold_ic["test_end"] == test_end)]["fold_id"].iloc[0]
    df_period["fold_id"] = which_fold     

    evidence_cols = []
    for factor in col_factor_qualified:
        value = fold_ic[(fold_ic["test_start"] == test_start) & (fold_ic["test_end"] == test_end)][f"weight_{factor}"].iloc[0]
        df_period[f"evidence_{factor}"] = value * df_period[factor]
        evidence_cols.append(f"evidence_{factor}")
    
    df_period["equal_weight"] = df_period[col_factor_qualified].mean(axis = 1)    
    df_period["evidence_weight"] = df_period[evidence_cols].sum(axis = 1)
    
    return df_period.drop(columns = evidence_cols)

build_composite_alpha(d1_df,"2023-01-01","2023-01-31",fold_ic, col_factor_qualified, "fwd_1d" ) 

,timestamp,symbol,in_universe,z_vol_14d,z_vol_of_vol_14d,fwd_1d,fold_id,equal_weight,evidence_weight
59629,2023-01-01 00:00:00+00:00,BTC/USDT,1,1.162525,0.707390,0.003372,1,0.934957,0.985098
59630,2023-01-01 00:00:00+00:00,ETH/USDT,1,0.921146,0.026073,0.011316,1,0.473610,0.572216
59631,2023-01-01 00:00:00+00:00,XRP/USDT,1,0.075388,1.009617,0.027662,1,0.542503,0.439582
59632,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.717912,0.571819,0.016380,1,-0.073047,-0.215131
59633,2023-01-01 00:00:00+00:00,BNB/USDT,1,0.569293,-0.303731,0.003268,1,0.132781,0.228959
...,...,...,...,...,...,...,...,...,...
65266,2023-01-31 00:00:00+00:00,QUICK/USDT,0,NaN,NaN,-0.001663,1,NaN,0.000000
65267,2023-01-31 00:00:00+00:00,DCR/USDT,0,NaN,NaN,0.030240,1,NaN,0.000000
65268,2023-01-31 00:00:00+00:00,STEEM/USDT,0,NaN,NaN,0.027823,1,NaN,0.000000
65269,2023-01-31 00:00:00+00:00,ONG/USDT,0,NaN,NaN,0.011158,1,NaN,0.000000


In [10]:
# ta đã có hàm tính thời gian test trong từng fold, giờ sẽ gộp từng fold đó
temp = []
for _, row in fold_ic.iterrows():
    test_start = row["test_start"]
    test_end = row["test_end"]
    
    df_period = build_composite_alpha(d1_df,test_start,test_end,fold_ic, col_factor_qualified, "fwd_1d" )
    temp.append(df_period)

df_ic_weight = pd.concat(temp, ignore_index= True)
df_ic_weight

,timestamp,symbol,in_universe,z_vol_14d,z_vol_of_vol_14d,fwd_1d,fold_id,equal_weight,evidence_weight
0,2023-01-01 00:00:00+00:00,BTC/USDT,1,1.162525,0.707390,0.003372,1,0.934957,0.985098
1,2023-01-01 00:00:00+00:00,ETH/USDT,1,0.921146,0.026073,0.011316,1,0.473610,0.572216
2,2023-01-01 00:00:00+00:00,XRP/USDT,1,0.075388,1.009617,0.027662,1,0.542503,0.439582
3,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.717912,0.571819,0.016380,1,-0.073047,-0.215131
4,2023-01-01 00:00:00+00:00,BNB/USDT,1,0.569293,-0.303731,0.003268,1,0.132781,0.228959
...,...,...,...,...,...,...,...,...,...
258042,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,0.024672,36,NaN,0.000000
258043,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,0.012474,36,NaN,0.000000
258044,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,0.017225,36,NaN,0.000000
258045,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,0.073563,36,NaN,0.000000


In [11]:
df_ic_weight[(df_ic_weight["timestamp"] == "2023-01-01 00:00:00+00:00") &(df_ic_weight["in_universe"]==1)]

,timestamp,symbol,in_universe,z_vol_14d,z_vol_of_vol_14d,fwd_1d,fold_id,equal_weight,evidence_weight
0,2023-01-01 00:00:00+00:00,BTC/USDT,1,1.162525,0.707390,0.003372,1,0.934957,0.985098
1,2023-01-01 00:00:00+00:00,ETH/USDT,1,0.921146,0.026073,0.011316,1,0.473610,0.572216
2,2023-01-01 00:00:00+00:00,XRP/USDT,1,0.075388,1.009617,0.027662,1,0.542503,0.439582
3,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.717912,0.571819,0.016380,1,-0.073047,-0.215131
4,2023-01-01 00:00:00+00:00,BNB/USDT,1,0.569293,-0.303731,0.003268,1,0.132781,0.228959
5,2023-01-01 00:00:00+00:00,LTC/USDT,1,0.189510,0.675351,0.052604,1,0.432431,0.378908
6,2023-01-01 00:00:00+00:00,SOL/USDT,1,-0.737448,0.753556,0.121447,1,0.008054,-0.156204
7,2023-01-01 00:00:00+00:00,EUR/USDT,1,1.630483,1.303691,-0.004477,1,1.467087,1.503088
8,2023-01-01 00:00:00+00:00,LINK/USDT,1,0.439154,0.376366,0.011132,1,0.407760,0.414677
9,2023-01-01 00:00:00+00:00,ADA/USDT,1,0.237101,-0.081475,0.015486,1,0.077813,0.112909


## D2: L2 regression

- Ta sử dụng model hôi quy từ 2 factor đã được chọn, để dự đoán fwd_1d  
  
- Phần model dự đoán sẽ là weight của phương pháp này

In [ ]:

d2_df = df[["timestamp","symbol","in_universe"] + col_factor_qualified + col_forward_return].copy()
d2_df.head()

,timestamp,symbol,in_universe,z_vol_14d,z_vol_of_vol_14d,fwd_1d,fwd_5d,fwd_14d
0,2022-01-01 00:00:00+00:00,BTC/USDT,1,-1.382781,-1.080754,-0.009188,-0.102294,-0.102248
1,2022-01-01 00:00:00+00:00,ETH/USDT,1,-1.499076,0.177704,0.016522,-0.100115,-0.124109
2,2022-01-01 00:00:00+00:00,BNB/USDT,1,-1.584325,-0.175895,0.006992,-0.109520,-0.064222
3,2022-01-01 00:00:00+00:00,LUNA/USDT,1,0.036502,1.591304,-0.024734,-0.156462,-0.050324
4,2022-01-01 00:00:00+00:00,SAND/USDT,1,1.646163,1.415158,-0.009780,-0.131208,-0.209213


In [14]:
fold.head()

,fold_id,train_start,train_end,test_start,test_end,ic_mean_z_vol_14d,ic_mean_z_vol_of_vol_14d,weight_z_vol_14d,weight_z_vol_of_vol_14d
0,1,2022-01-01 00:00:00+00:00,2022-12-31 00:00:00+00:00,2023-01-01,2023-01-31,0.123696,0.079029,0.610166,0.389834
1,2,2022-02-01 00:00:00+00:00,2023-01-31 00:00:00+00:00,2023-02-01,2023-02-28,0.104849,0.061353,0.630855,0.369145
2,3,2022-03-01 00:00:00+00:00,2023-02-28 00:00:00+00:00,2023-03-01,2023-03-31,0.103684,0.064275,0.617316,0.382684
3,4,2022-04-01 00:00:00+00:00,2023-03-31 00:00:00+00:00,2023-04-01,2023-04-30,0.11055,0.063878,0.633786,0.366214
4,5,2022-05-01 00:00:00+00:00,2023-04-30 00:00:00+00:00,2023-05-01,2023-05-31,0.10291,0.053385,0.658435,0.341565


In [15]:
fold_d2 = fold.drop(columns=["ic_mean_z_vol_14d","ic_mean_z_vol_of_vol_14d", 
                             "weight_z_vol_14d","weight_z_vol_of_vol_14d"])

fold_d2["train_start"] = pd.to_datetime(
    fold_d2["train_start"], utc=True
)

fold_d2["train_end"] = pd.to_datetime(
    fold_d2["train_end"], utc=True
)

fold_d2["test_start"] = pd.to_datetime(
    fold_d2["test_start"], utc=True
)

fold_d2["test_end"] = pd.to_datetime(
    fold_d2["test_end"], utc=True
)

fold_d2


,fold_id,train_start,train_end,test_start,test_end
0,1,2022-01-01 00:00:00+00:00,2022-12-31 00:00:00+00:00,2023-01-01 00:00:00+00:00,2023-01-31 00:00:00+00:00
1,2,2022-02-01 00:00:00+00:00,2023-01-31 00:00:00+00:00,2023-02-01 00:00:00+00:00,2023-02-28 00:00:00+00:00
2,3,2022-03-01 00:00:00+00:00,2023-02-28 00:00:00+00:00,2023-03-01 00:00:00+00:00,2023-03-31 00:00:00+00:00
3,4,2022-04-01 00:00:00+00:00,2023-03-31 00:00:00+00:00,2023-04-01 00:00:00+00:00,2023-04-30 00:00:00+00:00
4,5,2022-05-01 00:00:00+00:00,2023-04-30 00:00:00+00:00,2023-05-01 00:00:00+00:00,2023-05-31 00:00:00+00:00
5,6,2022-06-01 00:00:00+00:00,2023-05-31 00:00:00+00:00,2023-06-01 00:00:00+00:00,2023-06-30 00:00:00+00:00
6,7,2022-07-01 00:00:00+00:00,2023-06-30 00:00:00+00:00,2023-07-01 00:00:00+00:00,2023-07-31 00:00:00+00:00
7,8,2022-08-01 00:00:00+00:00,2023-07-31 00:00:00+00:00,2023-08-01 00:00:00+00:00,2023-08-31 00:00:00+00:00
8,9,2022-09-01 00:00:00+00:00,2023-08-31 00:00:00+00:00,2023-09-01 00:00:00+00:00,2023-09-30 00:00:00+00:00
9,10,2022-10-01 00:00:00+00:00,2023-09-30 00:00:00+00:00,2023-10-01 00:00:00+00:00,2023-10-31 00:00:00+00:00


In [16]:
def l2_regression_each_fold(df, train_start, train_end, test_start, test_end,
                            col_factor_qualified, target_col):

    alpha_values = 1.0
    
    train_period = df[(df["timestamp"] >= train_start) & (df["timestamp"] <= train_end) &(df["in_universe"] == 1)].copy()
    test_period = df[(df["timestamp"] >= test_start) & (df["timestamp"] <= test_end) &(df["in_universe"] == 1)].copy()
    

    x_train = train_period[col_factor_qualified].values
    y_train = train_period[target_col]
    
    x_test = test_period[col_factor_qualified].values
    
    
    if np.any(np.isnan(x_train)) or np.any(np.isnan(y_train)):
        mask_train = (~np.isnan(x_train).any(axis = 1)) & (~np.isnan(y_train))
        x_train = x_train[mask_train]
        y_train = y_train[mask_train]
        
        mask_test = ~np.isnan(x_test).any(axis=1)
        x_test = x_test[mask_test]
        test_period = test_period.loc[mask_test] 
    
    ridge = Ridge(alpha = alpha_values)
    ridge.fit(x_train,y_train)
    
    y_prediction = ridge.predict(x_test)
    temp = test_period[["timestamp","symbol"]].copy()
    temp["weight_ridge"] = y_prediction
    
    return temp

l2_regression_each_fold(d2_df,"2022-01-01 00:00:00+00:00", "2022-12-31 00:00:00+00:00", "2023-01-01", "2023-01-31", 
                        col_factor_qualified, "fwd_1d" )

,timestamp,symbol,weight_ridge
59629,2023-01-01 00:00:00+00:00,BTC/USDT,-0.002666
59630,2023-01-01 00:00:00+00:00,ETH/USDT,-0.003412
59631,2023-01-01 00:00:00+00:00,XRP/USDT,-0.005506
59632,2023-01-01 00:00:00+00:00,DOGE/USDT,-0.007680
59633,2023-01-01 00:00:00+00:00,BNB/USDT,-0.004397
...,...,...,...
65123,2023-01-31 00:00:00+00:00,ZIL/USDT,-0.005805
65124,2023-01-31 00:00:00+00:00,SUSHI/USDT,-0.005025
65125,2023-01-31 00:00:00+00:00,ALGO/USDT,-0.005257
65126,2023-01-31 00:00:00+00:00,UNI/USDT,-0.004428


In [17]:
def build_l2_regression(d2_df, fold, col_factor_qualified, horizon: str):
    fold = fold.copy()
    temp = []
    
    for _, row in fold.iterrows():
        train_start = row["train_start"]
        train_end = row["train_end"]
        test_start = row["test_start"]
        test_end = row["test_end"]
        l2_regression_this_fold = l2_regression_each_fold(d2_df, train_start=train_start, train_end= train_end, test_start=test_start,
                                                          test_end=test_end, col_factor_qualified = col_factor_qualified,
                                                          target_col= horizon)
        l2_regression_this_fold["fold_id"] = row["fold_id"] 
        temp.append(l2_regression_this_fold)
    result = pd.concat(temp, ignore_index= True)
    return pd.DataFrame(result)

df_ridge_weight = build_l2_regression(d2_df, fold_d2, col_factor_qualified, "fwd_1d")
df_ridge_weight

,timestamp,symbol,weight_ridge,fold_id
0,2023-01-01 00:00:00+00:00,BTC/USDT,-0.002666,1
1,2023-01-01 00:00:00+00:00,ETH/USDT,-0.003412,1
2,2023-01-01 00:00:00+00:00,XRP/USDT,-0.005506,1
3,2023-01-01 00:00:00+00:00,DOGE/USDT,-0.007680,1
4,2023-01-01 00:00:00+00:00,BNB/USDT,-0.004397,1
...,...,...,...,...
39985,2025-12-31 00:00:00+00:00,APT/USDT,-0.003996,36
39986,2025-12-31 00:00:00+00:00,LUNC/USDT,-0.009980,36
39987,2025-12-31 00:00:00+00:00,ARB/USDT,-0.003412,36
39988,2025-12-31 00:00:00+00:00,ICP/USDT,-0.005578,36


## D3: weight lightgbm

In [18]:
def lightgbm_each_fold(df, train_start, train_end, test_start, test_end,
                            col_factor_qualified, target_col, ratio = 0.2):

    """ 
    Ta chia train_period ra làm 2 phần; phần 1 là cho việc training, phần 2 cho phần validation để tìm ra bộ tham số phù hợp nhất cho model
    Logic triển khai tương tự như phần Ridge, ta sẽ chia train_period thành 2 phần gồm 80% và 20% data.
    """
    train_period = df[(df["timestamp"] >= train_start) & (df["timestamp"] <= train_end) &(df["in_universe"] == 1)].copy()
    test_period = df[(df["timestamp"] >= test_start) & (df["timestamp"] <= test_end) &(df["in_universe"] == 1)].copy()
    
    n = len(train_period)
    idx_train_val = int((1- ratio) *n) 

    x_train = train_period[col_factor_qualified].values
    y_train = train_period[target_col].values

    x_train_train = x_train[:idx_train_val] # tập x gồm 80% của tập x_train gốc
    y_train_train = y_train[:idx_train_val] # tập y gồm 80% của tập y_train gốc
    
    x_train_val = x_train[idx_train_val:] # tập x gồm 20% của tập x_val gốc
    y_train_val = y_train[idx_train_val:] # tập y gồm 20% của tập y_val gốc
    
    x_test = test_period[col_factor_qualified].values
        
    
    params = {
        "objective":"regression",
        "metric": "rmse",
        'boosting_type': 'gbdt',
        'num_leaves': 8,                
        'max_depth': 4,
        'min_data_in_leaf': 20,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'reg_alpha': 0.5,
        'reg_lambda': 0.5,
        'verbose': -1,
        'n_estimators': 1000,
    }
    
    train_lgb = lgb.Dataset( x_train_train, label = y_train_train)
    val_lgb = lgb.Dataset(x_train_val, label = y_train_val)
    
    model = lgb.train(params, train_lgb, valid_sets=[val_lgb], num_boost_round= 1000, callbacks= [lgb.early_stopping(stopping_rounds=50)])
    
    best_iter = model.best_iteration
    
    y_prediction = model.predict(x_test, num_iteration= best_iter)
    
    
    test_results = test_period[['timestamp', 'symbol']].copy()
    test_results['lightgbm_weight'] = y_prediction
    test_results['fold_id'] = 1  
    test_results['best_iteration'] = best_iter

    return test_results

lightgbm_each_fold(d2_df,"2022-01-01 00:00:00+00:00", "2022-12-31 00:00:00+00:00", "2023-01-01", "2023-01-31", 
                        col_factor_qualified, "fwd_1d" )



Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[6]	valid_0's rmse: 0.0535452


,timestamp,symbol,lightgbm_weight,fold_id,best_iteration
59629,2023-01-01 00:00:00+00:00,BTC/USDT,-0.005033,1,6
59630,2023-01-01 00:00:00+00:00,ETH/USDT,-0.004854,1,6
59631,2023-01-01 00:00:00+00:00,XRP/USDT,-0.005981,1,6
59632,2023-01-01 00:00:00+00:00,DOGE/USDT,-0.005981,1,6
59633,2023-01-01 00:00:00+00:00,BNB/USDT,-0.004854,1,6
...,...,...,...,...,...
65123,2023-01-31 00:00:00+00:00,ZIL/USDT,-0.006196,1,6
65124,2023-01-31 00:00:00+00:00,SUSHI/USDT,-0.005735,1,6
65125,2023-01-31 00:00:00+00:00,ALGO/USDT,-0.006417,1,6
65126,2023-01-31 00:00:00+00:00,UNI/USDT,-0.004854,1,6


In [19]:
def build_lighgbm(d2_df, fold, col_factor_qualified, horizon: str):
    fold = fold.copy()
    temp = []
        
    for _, row in fold.iterrows():
        train_start = row["train_start"]
        train_end = row["train_end"]
        test_start = row["test_start"]
        test_end = row["test_end"]
        lightgbm_this_fold = lightgbm_each_fold(d2_df, train_start=train_start, train_end= train_end, test_start=test_start,
                                                              test_end=test_end, col_factor_qualified = col_factor_qualified,
                                                              target_col= horizon)
        lightgbm_this_fold["fold_id"] = row["fold_id"] 
        temp.append(lightgbm_this_fold)
    result = pd.concat(temp, ignore_index= True)
    return pd.DataFrame(result)

df_lightgbm_weight = build_lighgbm(d2_df, fold_d2, col_factor_qualified, "fwd_1d")
df_lightgbm_weight

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[6]	valid_0's rmse: 0.0535452
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[19]	valid_0's rmse: 0.0566638
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[5]	valid_0's rmse: 0.0507137
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[8]	valid_0's rmse: 0.05829
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[5]	valid_0's rmse: 0.0495712
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's rmse: 0.0379785
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's rmse: 0.0412504
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[17]	valid_0's rmse: 

,timestamp,symbol,lightgbm_weight,fold_id,best_iteration
0,2023-01-01 00:00:00+00:00,BTC/USDT,-0.005033,1,6
1,2023-01-01 00:00:00+00:00,ETH/USDT,-0.004854,1,6
2,2023-01-01 00:00:00+00:00,XRP/USDT,-0.005981,1,6
3,2023-01-01 00:00:00+00:00,DOGE/USDT,-0.005981,1,6
4,2023-01-01 00:00:00+00:00,BNB/USDT,-0.004854,1,6
...,...,...,...,...,...
40644,2025-12-31 00:00:00+00:00,APT/USDT,-0.002415,36,5
40645,2025-12-31 00:00:00+00:00,LUNC/USDT,-0.002415,36,5
40646,2025-12-31 00:00:00+00:00,ARB/USDT,-0.002415,36,5
40647,2025-12-31 00:00:00+00:00,ICP/USDT,-0.002116,36,5


In [20]:
df_weight_summary = df_ridge_weight.merge(df_lightgbm_weight, on = ["timestamp","symbol","fold_id"], 
                                          how = "inner").merge(df_ic_weight[["timestamp","symbol","fold_id","equal_weight",
                                                                                "evidence_weight","fwd_1d"]], on = ["timestamp","symbol","fold_id"],
                                                               how = 'inner')

df_weight_summary


,timestamp,symbol,weight_ridge,fold_id,lightgbm_weight,best_iteration,equal_weight,evidence_weight,fwd_1d
0,2023-01-01 00:00:00+00:00,BTC/USDT,-0.002666,1,-0.005033,6,0.934957,0.985098,0.003372
1,2023-01-01 00:00:00+00:00,ETH/USDT,-0.003412,1,-0.004854,6,0.473610,0.572216,0.011316
2,2023-01-01 00:00:00+00:00,XRP/USDT,-0.005506,1,-0.005981,6,0.542503,0.439582,0.027662
3,2023-01-01 00:00:00+00:00,DOGE/USDT,-0.007680,1,-0.005981,6,-0.073047,-0.215131,0.016380
4,2023-01-01 00:00:00+00:00,BNB/USDT,-0.004397,1,-0.004854,6,0.132781,0.228959,0.003268
...,...,...,...,...,...,...,...,...,...
39985,2025-12-31 00:00:00+00:00,APT/USDT,-0.003996,36,-0.002415,5,-0.258591,-0.325419,0.116510
39986,2025-12-31 00:00:00+00:00,LUNC/USDT,-0.009980,36,-0.002415,5,-2.261695,-2.095396,-0.047119
39987,2025-12-31 00:00:00+00:00,ARB/USDT,-0.003412,36,-0.002415,5,0.044025,0.017039,0.073764
39988,2025-12-31 00:00:00+00:00,ICP/USDT,-0.005578,36,-0.002116,5,-0.943010,-1.038792,0.055123


## D4: Statistical significant between weight and horizon

In [ ]:
def build_ic_weight_horizon(df_weight_summary, weight_col,weight_base, target_col):
    temp = df_weight_summary.copy()
    result = []
    
    for timestamp, group in temp.groupby("timestamp"):
        weight_valux`e = group[weight_col].values
        target_value = group[target_col].values
        weight_base_value = group[weight_base].values
        if np.any(np.isnan(weight_value)) or np.any(np.isnan(target_value)):
            mask = ~(np.isnan(weight_value)) & ~(np.isnan(target_value))
            weight_value = weight_value[mask]
            target_value = target_value[mask]
        
        corr_base, p_value = spearmanr(weight_base_value, target_value)    
        corr, p_value = spearmanr(weight_value, target_value)
        result.append({
            "timestamp": timestamp,
            "ic_spearman": corr,
            "delta_ic": corr - corr_base
        })
    return pd.DataFrame(result)

build_ic_weight_horizon(df_weight_summary, "weight_ridge","equal_weight","fwd_1d")

,timestamp,ic_spearman,delta_ic
0,2023-01-01 00:00:00+00:00,-0.645257,-0.186759
1,2023-01-02 00:00:00+00:00,-0.253953,-0.007905
2,2023-01-03 00:00:00+00:00,0.115613,0.307312
3,2023-01-04 00:00:00+00:00,0.376623,0.250706
4,2023-01-05 00:00:00+00:00,-0.192547,-0.123094
...,...,...,...
1091,2025-12-27 00:00:00+00:00,0.344916,-0.006950
1092,2025-12-28 00:00:00+00:00,0.499614,0.010039
1093,2025-12-29 00:00:00+00:00,0.452661,-0.004202
1094,2025-12-30 00:00:00+00:00,0.195798,0.008683


In [22]:
def compute_newey_west_tstat(series, max_lags=4):

    y = series.dropna().values
    n = len(y)

    X = np.ones((n, 1))

    model = OLS(y, X).fit(
        cov_type='HAC',
        cov_kwds={'maxlags': max_lags}
    )

    t_stat = model.tvalues[0]
    p_value = model.pvalues[0]

    return t_stat, p_value

In [23]:
weight_factor = ["weight_ridge","lightgbm_weight","equal_weight","evidence_weight"]

In [24]:
def statistics_sig_ic_weight(df_weight_summary, weight_factor, horizon, weight_base):
    temp = df_weight_summary.copy()
    result = []
    
    for weight in weight_factor:
        significant = None
        ic_weight_horizon = build_ic_weight_horizon(temp, weight, weight_base, horizon)
        ic_mean = ic_weight_horizon["ic_spearman"].mean()
        t_stat, p_value = compute_newey_west_tstat(ic_weight_horizon["delta_ic"])
        
        if p_value < 0.01:
            significant = "1% alpha"
        else:
            significant = "not significant"
        result.append({
            "weight_type": weight,
            "ic_mean": ic_mean,
            "delta_ic_mean": ic_weight_horizon["delta_ic"].mean(),
            "t_stat": t_stat,
            "p_value": p_value,
            "sig_delta_ic": significant
        })
    return pd.DataFrame(result)

statistics_sig_ic_weight(df_weight_summary,weight_factor,"fwd_1d","equal_weight")
        

,weight_type,ic_mean,delta_ic_mean,t_stat,p_value,sig_delta_ic
0,weight_ridge,0.069358,-0.006174,-2.593171,9.509539e-03,1% alpha
1,lightgbm_weight,0.021750,-0.053782,-7.016979,2.267159e-12,1% alpha
2,equal_weight,0.075533,0.000000,NaN,NaN,not significant
3,evidence_weight,0.076633,0.001101,1.413986,1.573659e-01,not significant


Qua phần kết quả phía trên, ta có thể có vài đánh giá như sau: 
- Cả ridge và lightgbm đều thua so với baseline weight. Vì mục tiêu là xếp hạng coin nên ta tính chỉ số coeff spearmanr, khi đó ta cần biết hiệu coeff_mean của weight ta đang xét so với equal weight (lầ delta_ic_mean) có lớn hơn 0 và có ý nghĩa thống kê hay không (vì ta để coeff_weight - coeff_baseline)

- Evidence weight có delta_ic_mean lớn hơn 0 tuy nhiên lại không qua được kiểm định thống kê dù chỉ mức 5%, nên có thể suy ra rằng đó là do nhiễu ngẫu nhiên

- Vậy là evidence_weight vẫn là một lựa chọn tốt, và sẽ là weight chính để backtest phía sau

In [25]:
last_df_weight_summary = df_weight_summary[["timestamp","symbol","fold_id","equal_weight","evidence_weight","fwd_1d"]]
last_df_weight_summary

,timestamp,symbol,fold_id,equal_weight,evidence_weight,fwd_1d
0,2023-01-01 00:00:00+00:00,BTC/USDT,1,0.934957,0.985098,0.003372
1,2023-01-01 00:00:00+00:00,ETH/USDT,1,0.473610,0.572216,0.011316
2,2023-01-01 00:00:00+00:00,XRP/USDT,1,0.542503,0.439582,0.027662
3,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.073047,-0.215131,0.016380
4,2023-01-01 00:00:00+00:00,BNB/USDT,1,0.132781,0.228959,0.003268
...,...,...,...,...,...,...
39985,2025-12-31 00:00:00+00:00,APT/USDT,36,-0.258591,-0.325419,0.116510
39986,2025-12-31 00:00:00+00:00,LUNC/USDT,36,-2.261695,-2.095396,-0.047119
39987,2025-12-31 00:00:00+00:00,ARB/USDT,36,0.044025,0.017039,0.073764
39988,2025-12-31 00:00:00+00:00,ICP/USDT,36,-0.943010,-1.038792,0.055123


In [26]:
last_df_weight_summary.to_parquet('D1_df_weightt.parquet')